<a href="https://colab.research.google.com/github/RPGNZ/MA-DCA-Strategy/blob/main/Backtest_MA_DCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moving average DCA strategy vs. regular DCA

### Disclaimer
The Python code presented here is for educational and informational purposes only. It is not intended as investment advice or as a recommendation to buy, sell or hold any particular asset. The author is not a financial advisor and is not responsible for any investment decisions made by the reader. The reader should conduct their own research and analysis before making any investment decisions. The author is not liable for any losses or damages that may arise from the use of this code.

### Imports

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import math
!pip install numpy-financial
import numpy_financial as npf

### Functions

#### Monthly MA DCA

In [24]:
def monthly_MA_DCA(daily_data: pd.DataFrame, cash: float, allow_fractional_shares: bool, MA_DCA_params):
    # Buy 'cash' amount of the asset at the open level every first day of the month
    # depending on MA conditions

    dca_history = []
    order_size = 0           # this month order size
    position_size = 0        # cumulative position size
    total_cash_invested = 0  # total amount of cash allocated to the strategy
    total_invested = 0       # current total amount of cash invested to buy the asset
    cash_dca = 0             # current amount of cash available
    executed_orders = []     # All executed orders

    data = daily_data.copy() # creates a copy of daily_data dataframe

    # Compute daily EMA to be used as strategy arbitration
    data['MA'] = data['Close'].ewm(span=MA_DCA_params["period"], adjust=False).mean()

    data['Order_size'] = float('nan')          # Size of current month order
    data['Invested'] = 0.0                     # Amount spent to buy assets
    data['Available_cash'] = float('nan')      # Remaing cash after the order is executed
    data['Total_cash_invested'] = float('nan') # Total amount of cash put in the DCA strategy

    prev_day_month = data.index[0].month

    for date, row in data.iloc[1:].iterrows():
        order_size = 0
        open_price = row['Open']
        ma_prev_day = data.loc[data.index < date, 'MA'].iloc[-1]

        if date.month != prev_day_month: # at first day of the new month
            prev_day_month = date.month

            # add monthly cash to the strategy
            cash_dca += cash
            total_cash_invested += cash

            if open_price > ma_prev_day:
                # Above MA: buy monthly amount + 50% of the cash accumulated during the previous bearish period
                if allow_fractional_shares:
                    order_size = (cash + 0.5*(cash_dca-cash)) / open_price
                else:
                    order_size = np.floor((cash + 0.5*(cash_dca-cash)) / open_price)
            else:
                # below MA: buy only 20% of monthly amount
                if allow_fractional_shares:
                    order_size = 0.2*cash / open_price
                else:
                    order_size = np.floor(0.5*cash / open_price)

            executed_orders.append({
                    'exec_day': date,
                    'price': open_price,
                    'invested': order_size * open_price,
                    'order_size': order_size
                    })

        cash_dca -= order_size * open_price
        position_size  += order_size
        total_invested += order_size * open_price
        data.at[date, 'Order_size'] = order_size
        data.at[date, 'Position'] = position_size
        data.at[date, 'Invested'] = order_size*open_price
        data.at[date, 'Total_invested'] = total_invested
        data.at[date, 'Available_cash'] = cash_dca
        data.at[date, 'Total_cash_invested'] = total_cash_invested

    data['Portfolio_value'] = data['Position'] * data['Close'] + data['Available_cash']
    data['Average_buy_price'] = data['Total_invested'] / data['Position']

    executed_orders_df = pd.DataFrame(executed_orders)

    return data, executed_orders_df

#### Monthly DCA

In [18]:
def monthly_DCA(daily_data: pd.DataFrame, cash: float, allow_fractional_shares: bool):
    # Buy 'cash' amount of the asset at the open level every first day of the month

    dca_history = []
    order_size = 0           # this month order size
    position_size = 0        # cumulative position size
    total_cash_invested = 0  # total amount of cash allocated to the strategy
    cash_dca = 0             # current amount of cash available

    data = daily_data.copy() # creates a copy of daily_data dataframe

    data['Order_size'] = float('nan')          # Size of current month order
    data['Invested'] = 0.0                     # Amount spent to buy assets
    data['Available_cash'] = float('nan')      # Remaing cash after the order is executed
    data['Total_cash_invested'] = float('nan') # Total amount of cash put in the DCA strategy

    prev_day_month = data.index[0].month

    for date, row in data.iloc[1:].iterrows():
        if date.month != prev_day_month: # test whether we've started a new month
            prev_day_month = date.month
            total_cash_invested += cash

            open_price = row['Open']
            if allow_fractional_shares:
                order_size = cash / open_price
            else:
                cash_dca += cash
                order_size = np.floor(cash_dca / open_price)
                cash_dca -= order_size * open_price

            data.at[date, 'Order_size'] = order_size
            data.at[date, 'Invested'] = order_size*open_price
            data.at[date, 'Available_cash'] = cash_dca
            data.at[date, 'Total_cash_invested'] = total_cash_invested

    data['Total_invested'] = data['Invested'].cumsum()
    data['Position'] = data['Order_size'].cumsum().ffill().fillna(0)
    data['Available_cash'] = data['Available_cash'].ffill().fillna(0)
    data['Portfolio_value'] = data['Position'] * data['Close'] + data['Available_cash']
    data['Total_cash_invested'] = data['Total_cash_invested'].ffill().fillna(0)
    data['Average_buy_price'] = data['Total_invested'] / data['Position']

    return data

#### Display

In [4]:
def plot_ma_dca_backtest(
    data: pd.DataFrame,
    df_ma_dca: pd.DataFrame,
    df_dca: pd.DataFrame,
    executed_orders: pd.DataFrame,
    MA_DCA_params: dict,
    ticker: str
):
    # Plot comparison between MA-DCA strategy and regular DCA strategy.
    #
    # Parameters
    # ----------
    # data:                 Raw OHLC price data
    # df_ma_dca:            Strategy dataframe for MA-DCA
    # df_dca:               Strategy dataframe for regular DCA
    # executed_orders:      MA-DCA executed orders dataframe
    # MA_DCA_params:        Strategy parameters (EMA period...)
    # ticker:               Asset ticker symbol

    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    vertical_spacing=0.01,
                    # row_heights=[0.7, 0.1, 0.1, 0.1],
                    row_heights=[0.4, 0.2, 0.2, 0.2],
                    )


    ## Top part
    # ----------
    # - Asset candle bars
    # - MA
    # - MA-based strategy executed orders

    # Candle bars
    fig.add_trace(go.Candlestick(
        x=data.index,
        open=data['Open'],
        high=data['High'],
        low=data['Low'],
        close=data['Close'],
        name=f"{ticker}",
        increasing_line_color='green',
        decreasing_line_color='red',
        opacity=0.9
    ), row=1, col=1)


    # MA
    fig.add_trace(
        go.Scatter(
            x = df_ma_dca_fun.index,
            y = df_ma_dca_fun['MA'],
            mode='lines',
            name=f'MA {MA_DCA_params["period"]}',
            line=dict(color='yellow', width=2)
        ),
        row=1, col=1
    )


    # Executed orders
    if not executed_orders.empty:
        exec_orders = executed_orders[executed_orders['order_size'] > 0].copy()
        fig.add_trace(
            go.Scatter(
                x = exec_orders['exec_day'],
                y = exec_orders['price'],
                mode = 'markers',#'markers+text',
                name = 'Executed orders',
                marker = dict(size=12, color='lime', symbol='triangle-up',line=dict(width=1,color='White')),
                text = exec_orders['order_size'],
                textposition = 'top center',
                customdata = exec_orders[['order_size', 'invested']].values,
                hovertemplate = '<br>Date: %{x}<br>Price: %{y:.2f}<br>Size: %{customdata[0]:.4f}<br>Invested: %{customdata[1]:.2f}<extra></extra>'
            ),
            row=1, col=1
        )
        exec_orders_sell = executed_orders[executed_orders['order_size'] < 0].copy()
        fig.add_trace(
            go.Scatter(
                x = exec_orders_sell['exec_day'],
                y = exec_orders_sell['price'],
                mode = 'markers',
                name = 'Executed sell orders',
                marker = dict(size=12, color='red', symbol='triangle-down',line=dict(width=1,color='White')),
                text = exec_orders_sell['order_size'],
                textposition = 'bottom center',
                customdata = exec_orders_sell[['order_size', 'invested']].values,
                hovertemplate = '<br>Date: %{x}<br>Price: %{y:.2f}<br>Size: %{customdata[0]:.4f}<br>Invested: %{customdata[1]:.2f}<extra></extra>'
            ),
            row=1, col=1
        )


    ## Comparison part
    # ----------
    # - Positions
    # - Portfolio value
    # - Average entry price

    # MA DCA
    fig.add_trace( go.Scatter(
        x=df_ma_dca_fun.index,
        y = df_ma_dca_fun['Position'],
        # y = df_pivot_dca_fun['Portfolio_value'],
        # y = df_pivot_dca_fun['Available_cash'],
        # y= df_pivot_dca_fun['Average_buy_price'],
        mode='lines',
        name='MA-DCA (Monthly)',
        line=dict(color='orange', width=2)
    ),row=2, col=1)

    fig.add_trace( go.Scatter(
        x=df_ma_dca_fun.index,
        # y = df_pivot_dca_fun['Position'],
        y = df_ma_dca_fun['Portfolio_value'],
        # y = df_pivot_dca_fun['Available_cash'],
        # y= df_pivot_dca_fun['Average_buy_price'],
        mode='lines',
        name='MA-DCA (Monthly)',
        showlegend=False,
        line=dict(color='orange', width=2)
    ),row=3, col=1)

    fig.add_trace( go.Scatter(
        x=df_ma_dca_fun.index,
        # y = df_pivot_dca_fun['Position'],
        # y = df_pivot_dca_fun['Portfolio_value'],
        # y = df_pivot_dca_fun['Available_cash'],
        y= df_ma_dca_fun['Average_buy_price'],
        mode='lines',
        name='MA-DCA (Monthly)',
        showlegend=False,
        line=dict(color='orange', width=2)
    ),row=4, col=1)


    # DCA
    fig.add_trace(go.Scatter(
        x=df_dca_fun.index,
        y = df_dca_fun['Position'],
        # y = df_dca_fun['Portfolio_value'],
        # y = df_dca_fun['Available_cash'],
        # y = df_dca_fun['Average_buy_price'],
        mode='lines',
        name='DCA (Monthly)',
        line=dict(color='cyan', width=2)
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=df_dca_fun.index,
        # y = df_dca_fun['Position'],
        y = df_dca_fun['Portfolio_value'],
        # y = df_dca_fun['Available_cash'],
        # y = df_dca_fun['Average_buy_price'],
        mode='lines',
        name='DCA (Monthly)',
        showlegend=False,
        line=dict(color='cyan', width=2)
    ), row=3, col=1)

    fig.add_trace(go.Scatter(
        x=df_dca_fun.index,
        # y = df_dca_fun['Position'],
        # y = df_dca_fun['Portfolio_value'],
        # y = df_dca_fun['Available_cash'],
        y = df_dca_fun['Average_buy_price'],
        mode='lines',
        name='DCA (Monthly)',
        showlegend=False,
        line=dict(color='cyan', width=2)
    ), row=4, col=1)


    # Layout
    fig.update_yaxes(row=1, col=1, range=[min(data['Low'])*0.99, max(data['High'])*1.01])
    fig.update_yaxes(title_text="Position", row=2, col=1)
    fig.update_yaxes(title_text="Portfolio val.", row=3, col=1)
    fig.update_yaxes(title_text="Av. entry price", row=4, col=1)

    fig.update_layout(
        template='plotly_dark',
        title=f"📊 Moving Average based DCA vs. DCA (Monthly) — {ticker}",
        height=750,
        # width=800,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    fig.show()

#### Misc

In [5]:
def monthly_dca_irr(monthly_budget: float,
                    n_months: int,
                    final_value: float):
    # Computes the IRR of a monthly DCA strategy with constant contributions.
    #
    # Parameters
    # ----------
    # monthly_budget: Amount invested every month (negative cashflow).
    # n_months:       Number of months
    # final_value:    Final portfolio value at the end (positive cashflow).
    #
    # Returns
    # -------
    # monthly_irr:    Monthly internal rate of return.
    # annualized_irr: Annualized IRR equivalent (compounded).

    # Cashflows: monthly investments + final portfolio liquidation
    cashflows = [-monthly_budget] * n_months
    cashflows.append(final_value)

    # Monthly IRR
    monthly_irr = npf.irr(cashflows)

    # Annualize (12 months per year)
    annualized_irr = (1 + monthly_irr)**12 - 1

    return monthly_irr, annualized_irr

### Parameters

In [21]:
ticker = "QQQ" # NASDAQ-100
# ticker = "SPY" # S&P500
# ticker = "IWM" # Russel 2000
# ticker = "URTH" # MSCI World
# ticker = "GLD" # Gold
# ticker = "EXS1" # DAX
# ticker = "FXI" # iShares China Large-Cap
# ticker = "USO" # United States Oil Fund
# ticker = "BTC-USD"
# ticker = "AAPL"
# ticker = "TTE.PA" # TotalEnergies
# ticker = "RNL.F" # Renault

start_date = "2021-03-25" #"2025-10-26"
monthly_budget = 1000  # monthly budget to be invested
allow_fractional_shares = False # buy fractional shares of the asset

In [7]:
# MA DCA parameters
MA_DCA_params = {
    "ma_filter": True, # Use MA position for the strategy
    "period": 200, # MA period
}

### Backtest

In [25]:
# =====================
# 1. download daily data
# =====================
# Download data starting at Monday before start_date for accurate pivot level computation of first week
start_date = pd.to_datetime(start_date)
# start_monday = start_date - pd.Timedelta(days=start_date.weekday())

data = yf.download(ticker, start=start_date, interval="1d", auto_adjust=True)

# if the columns have a MultiIndex, flatten them.
if isinstance(data.columns, pd.MultiIndex):
    data.columns = [col[0] for col in data.columns]


# =====================
# 2. Execute and compare strategies
# =====================

# execute MA-based strategy
[df_ma_dca_fun, executed_orders] = monthly_MA_DCA(data, monthly_budget, allow_fractional_shares, MA_DCA_params)

total_cash_invested_ma = df_ma_dca_fun['Total_cash_invested'].iloc[-1]
final_portfolio_value_ma = df_ma_dca_fun['Portfolio_value'].iloc[-1]

perf_ma = final_portfolio_value_ma / total_cash_invested_ma - 1

w_irr, ann_irr_ma = monthly_dca_irr(monthly_budget, int(total_cash_invested_ma/monthly_budget), final_portfolio_value_ma)


# execute DCA strategy
df_dca_fun = monthly_DCA(data, monthly_budget, allow_fractional_shares)

total_cash_invested_dca = df_dca_fun['Total_cash_invested'].iloc[-1]
final_portfolio_value_dca = df_dca_fun['Portfolio_value'].iloc[-1]

perf_dca = final_portfolio_value_dca / total_cash_invested_dca - 1

w_irr, ann_irr_dca = monthly_dca_irr(monthly_budget, int(total_cash_invested_dca/monthly_budget), final_portfolio_value_dca)

[*********************100%***********************]  1 of 1 completed


### Results

In [26]:
print("\n=== 📊 MA-Based DCA vs DCA ===")
print(f"\n--- MA-based DCA ---")
print(f"Total cash invested: {df_ma_dca_fun['Total_cash_invested'].iloc[-1]:,.0f}")
print(f"Final portfolio value: {final_portfolio_value_ma:,.0f}")
print(f"Total return: {perf_ma:.2%}")
print(f"Annualized IRR: {ann_irr_ma:.2%}")
print(f"{ticker} final position: {df_ma_dca_fun['Position'].iloc[-1]:.4f}")
print(f"Remaining cash : {df_ma_dca_fun['Available_cash'].iloc[-1]:.2f}")


print(f"\n--- DCA ---")
print(f"Total cash invested: {df_dca_fun['Total_cash_invested'].iloc[-1]:,.0f}")
print(f"Final portfolio value: {df_dca_fun['Portfolio_value'].iloc[-1]:,.0f}")
print(f"Total return: {perf_dca:.2%}")
print(f"Annualized IRR: {ann_irr_dca:.2%}")
print(f"{ticker} final position: {df_dca_fun['Position'].iloc[-1]:.4f}")
print(f"Remaining cash : {df_dca_fun['Available_cash'].iloc[-1]:.2f}")

plot_ma_dca_backtest(
    data=data,
    df_ma_dca=df_ma_dca_fun,
    df_dca=df_dca_fun,
    executed_orders=executed_orders,
    MA_DCA_params=MA_DCA_params,
    ticker=ticker
)


=== 📊 MA-Based DCA vs DCA ===

--- MA-based DCA ---
Total cash invested: 60,000
Final portfolio value: 85,132
Total return: 41.89%
Annualized IRR: 13.96%
QQQ final position: 150.0000
Remaining cash : 745.38

--- DCA ---
Total cash invested: 60,000
Final portfolio value: 87,034
Total return: 45.06%
Annualized IRR: 14.86%
QQQ final position: 154.0000
Remaining cash : 396.58
